# Libraries

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import os

# Game table extraction logic

In [ ]:
def order_corners(pts):
    # receives 4 points = (x, y coordinates) = board corners 
    # the goal is to order them 
    rect = np.zeros((4, 2), dtype="float32")
    # sum x + y for each point
    # min sum = top-left
    # max sum = bottom-right
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]

    # diff x - y for each point
    # dim min = top-right corner
    # diff max = bottom-left corner
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    
    # rect = the 4 ordered corners
    return rect

def extract_game_table(image_path):
    # read the image 
    img = cv.imread(image_path)
    if img is None:
        print(f"The image couldn't be extracted. {image_path}")
        return None

    # height/width 
    # total number of pixels for filtering
    h_img, w_img = img.shape[:2]
    total_pixels = h_img * w_img

    # from BGR to HSV
    # better for colors
    hsv = cv.cvtColor(img, cv.COLOR_BGR2HSV)
    
    # green range in HSV
    lower_green = np.array([30, 35, 35])
    upper_green = np.array([90, 255, 255])
    
    # binary mask for green
    mask = cv.inRange(hsv, lower_green, upper_green)
    
    # I used morphologyEx to extract the image, I tried other variants with Canny 
    # but it didn't work well if I had pieces over the edges
    # kernel 60,60 = how big the filling area is
    # morph-close = fills the holes in the mask
    # 3 iterations, this way I almost get a completely white square, very easy to extract
    kernel = np.ones((60, 60), np.uint8)
    mask = cv.morphologyEx(mask, cv.MORPH_CLOSE, kernel, iterations=3)
    
    # removes noise 
    mask_blur = cv.GaussianBlur(mask, (5, 5), 0)
    
    # laplacian for edge detection
    # cv.CV_64F I used it to extract better diff, I didn't lose information (negative numbers)
    # then I converted them back to 8 bits
    laplacian = cv.Laplacian(mask_blur, cv.CV_64F)
    laplacian = cv.convertScaleAbs(laplacian)
    
    # binary image based on a threshold = 20 
    _, edges = cv.threshold(laplacian, 20, 255, cv.THRESH_BINARY)
    
    # dilate edges = thickens them
    # for a more complete contour
    edges = cv.dilate(edges, np.ones((3,3), np.uint8), iterations=2)

    # find contours 
    # retr_external gives only external contours 
    contours, _ = cv.findContours(edges, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    
    max_area = 0
    best_corners = None
    # img_debug = img.copy()

    # iterate through all found contours
    for cnt in contours:
        area = cv.contourArea(cnt)
        
        # filter based on area
        # ignore noisy areas or areas that occupy almost the whole image
        if area > 20000 and area < (total_pixels * 0.95):
            
            # calculate hull on the current contour, takes all and makes a square 
            hull = cv.convexHull(cnt)
            
            # minimum area rectangle 
            rect = cv.minAreaRect(hull)
            
            # the 4 corners of the board 
            box = cv.boxPoints(rect)
            box = np.int32(box)
            
            # area
            rect_area = rect[1][0] * rect[1][1]
            
            # check area and keep the largest one
            if rect_area > max_area:
                max_area = rect_area
                
                # order corners, helper function
                best_corners = order_corners(box)
                
                # debug 
                # cv.drawContours(img_debug, [cnt], -1, (255, 0, 0), 2)
                # cv.drawContours(img_debug, [hull], -1, (0, 255, 255), 2)
                # for pt in box:
                #     cv.circle(img_debug, tuple(pt), 15, (0, 255, 0), -1)
                # cv.drawContours(img_debug, [box], 0, (0, 255, 0), 3)

    # check if we found anything  
    if best_corners is None:
        print("The game board couldn't be found.")
        return None
    
    # final dim without padding
    board_w = 1600
    board_h = 1600
    
    # add padding for template matching 
    # padding is on both sides, so 1800x1800
    pad = 100
    final_w = board_w + 2*pad
    final_h = board_h + 2*pad
    
    # define corners with padding
    # order as in order_corners
    destination_corners = np.array([
        [pad, pad],                     # top left
        [pad + board_w, pad],           # top right
        [pad + board_w, pad + board_h], # bottom right
        [pad, pad + board_h]            # bottom left
    ], dtype="float32")

    # perspective transformation matrix
    M = cv.getPerspectiveTransform(best_corners, destination_corners)
    
    # apply the transformation on the original image
    # the result is the extracted board with padding
    result = cv.warpPerspective(img, M, (final_w, final_h))
    
    return result

# ##################################
# Debug
# ##################################
# path = 'antrenare/1_16.jpg' 
# if os.path.exists(path):
#     table = extract_game_table(path)
#     if table is not None:
#         plt.figure(figsize=(9,9))
#         plt.imshow(cv.cvtColor(table, cv.COLOR_BGR2RGB))
#         plt.title("Game Table:")
#         plt.axis('off')
#         plt.show()
# else:
#     print("The file does not exist.")

# Pieces placement/color detecting logic

In [ ]:
TEMPLATE_SIZE = 100
# template size
MATCH_THRESHOLD = 0.8 
# template matching threshold

def get_pieces_templates():
    # dictionary for templates
    templates = {}
    # iterate from 1 to 6 (piece types)
    for i in range(1, 7):
        # construct path to file (folder 'templates' must exist)
        path = f"templates/{i}.jpg"
        if os.path.exists(path):
            # image directly in grayscale 
            img = cv.imread(path, cv.IMREAD_GRAYSCALE)
            # resize to template size defined above, temps are already 100x100
            img = cv.resize(img, (TEMPLATE_SIZE, TEMPLATE_SIZE))
            templates[i] = img
    return templates

def get_color_simple(hsv_image, x, y, w, h):
    # detect average color hsv/piece center
    # coordinates of the piece center
    cx = x + w // 2
    cy = y + h // 2
    
    # ensure we don't go out of dimensions 
    cy = min(max(0, cy), hsv_image.shape[0]-1)
    cx = min(max(0, cx), hsv_image.shape[1]-1)
    
    # a small 10x10 pixel patch from the piece center
    # for average color
    img_center = hsv_image[cy-5:cy+5, cx-5:cx+5]
    
    # in case img_center is empty
    if img_center.size == 0: return '?'
    
    # mean hue and saturation 
    avg_h = np.mean(img_center[:, :, 0])
    avg_s = np.mean(img_center[:, :, 1])
    
    # color rules
    # s too small = white
    if avg_s < 40: return 'W'
    
    # from here hue
    # orange
    if 4 < avg_h <= 22: return 'O'
    # yellow
    if 22 < avg_h <= 35: return 'Y'
    # green
    if 35 < avg_h <= 85: return 'G'
    # blue
    if 85 < avg_h <= 135: return 'B'
    # red, here it is circular (like in lab 7)
    if avg_h <= 10 or avg_h >= 170: return 'R'
    
    # ? if classification fails
    return '?'

def find_all_pieces_on_board_and_color(image, templates):

    # convert to grayscale 
    gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
    
    # blur to get rid of noise 
    gray = cv.GaussianBlur(gray, (5, 5), 0) 
    
    # hsv for color, different image!!
    hsv = cv.cvtColor(image, cv.COLOR_BGR2HSV)

    # final matrix for pieces
    # ' ' - empty
    pieces_matrix = np.full((16, 16), '  ', dtype='object')
    
    # aux matrix to keep max score found in each cell
    # to verify results and keep the best ones
    pieces_scores = np.zeros((16, 16))
    
    # padding for board extraction 
    PAD_X = 100
    PAD_Y = 100
    
    # for debug ignore
    # img_viz = image.copy()
    
    # Template matching logic 
    # define scales for resizing
    # this is why code takes long, but without it it doesn t work well
    # scale template from 90% to 110% with 5% increments
    scales = np.linspace(0.9, 1.1, 5)

    # iterate through pieces
    for shape_id, base_tmpl in templates.items():
        
        # iterate through each scale 
        for scale in scales:
            # calculate new dimensions 
            new_w = int(base_tmpl.shape[1] * scale)
            new_h = int(base_tmpl.shape[0] * scale)
            
            # resize base template
            scaled_tmpl = cv.resize(base_tmpl, (new_w, new_h))
            
            # keep current dimensions to calculate center later
            curr_w, curr_h = scaled_tmpl.shape[::-1]

            # iterate through 4 rotations of the template
            for angle in [0, 90, 180, 270]:
                if angle == 0: 
                    t = scaled_tmpl
                else: 
                    # rotation functions from openCV
                    t = cv.rotate(scaled_tmpl, {
                        90: cv.ROTATE_90_CLOCKWISE, 
                        180: cv.ROTATE_180, 
                        270: cv.ROTATE_90_COUNTERCLOCKWISE
                    }[angle])
                
                # check if template is larger than image
                # if so, matchtemplate errors 
                if t.shape[0] > gray.shape[0] or t.shape[1] > gray.shape[1]:
                    continue
                
                # template matching
                # returns a score map between -1/1
                res = cv.matchTemplate(gray, t, cv.TM_CCOEFF_NORMED)
                
                # check > thresh/0.8
                locs = np.where(res >= MATCH_THRESHOLD)
                
                # iterate through found points
                for pt in zip(*locs[::-1]):
                    # extract score for current point
                    score = res[pt[1], pt[0]]
                    
                    # calculate piece center in image
                    # pt[0] is x (top left), pt[1] is y (top)
                    center_x = pt[0] + curr_w // 2
                    center_y = pt[1] + curr_h // 2
                    
                    # map pixel coordinates to matrix coordinates
                    # subtract padding and divide by cell size (100)
                    col = (center_x - PAD_X) // 100
                    row = (center_y - PAD_Y) // 100
                    
                    # check if we are on the game board
                    if 0 <= row < 16 and 0 <= col < 16:
                        # max logic:
                        # if we find a higher score than previously found, update info
                        if score > pieces_scores[row, col]:
                            pieces_scores[row, col] = score
                        
                            # detect color     
                            # send hsv and coordinates 
                            color = get_color_simple(hsv, pt[0], pt[1], curr_w, curr_h)
                            
                            # save final piece 
                            pieces_matrix[row, col] = f"{shape_id}{color}"
                            
                            # here for debug ignore
                            # cv.rectangle(img_viz, (pt[0], pt[1]), (pt[0]+curr_w, pt[1]+curr_h), (0, 255, 255), 1)

    # here mostly debug ignore
    # img_final = image.copy()
    # # draw on image 
    # for r in range(16):
    #     for c in range(16):
    #         if pieces_matrix[r, c] != '  ':
    #             # recalculate pos in pixels 
    #             x = PAD_X + c * 100 
    #             y = PAD_Y + r * 100
                
    #             # text on piece
    #             cv.putText(img_final, pieces_matrix[r, c], (x+10, y + 90), 
    #                        cv.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 4)
    #             # contour 
    #             cv.rectangle(img_final, (x, y), (x+100, y+100), (0, 255, 0), 8)

    # display
    # plt.figure(figsize=(9, 9))
    # plt.imshow(cv.cvtColor(img_final, cv.COLOR_BGR2RGB))
    # plt.title("Result 16x16")
    # plt.axis('off')
    # plt.show()

    # return detected pieces matrix
    return pieces_matrix

# Bonus detecting logic

In [ ]:
CELL_SIZE = 100 
# cell size for templates
CROP_MARGIN = 15 
# crop for bonus templates 15px 
MATCH_THRESHOLD = 0.70 
# matching threshold

# function for binary image 
def get_binary_image(img_gray):
    # adaptive gaussian thresholding
    # unlike simple thresholding, it calculates a local threshold
    bin_img = cv.adaptiveThreshold(img_gray, 255, cv.ADAPTIVE_THRESH_GAUSSIAN_C, cv.THRESH_BINARY_INV, 15, 3)
    
    # small kernel
    kernel = np.ones((2,2), np.uint8)
    # apply morphologyEx just like when extracting the board
    # removes small white dots and keeps large features, numbers
    bin_img = cv.morphologyEx(bin_img, cv.MORPH_OPEN, kernel)
    
    # return the binary image 
    return bin_img

# function for bonus 2 template
def load_bonus_2():
    # dictionary for template 
    templates = {}
    # path
    path = 'templates/bonus_2.jpg' 
    
    if os.path.exists(path):
        # read grayscale
        img = cv.imread(path, cv.IMREAD_GRAYSCALE)
        # resize
        img = cv.resize(img, (CELL_SIZE, CELL_SIZE))
        
        # crop by 15 pixels, I used Paint to measure
        # we get better accuracy without the border
        # this relates more to the first method I thought of but scrapped...
        img_crop = img[CROP_MARGIN : CELL_SIZE-CROP_MARGIN, CROP_MARGIN : CELL_SIZE-CROP_MARGIN]
        
        # transform to binary
        bin_tmpl = get_binary_image(img_crop)
        templates[2] = bin_tmpl
    else:
        print(f"No bonus template.{path}")
    
    # return the templates
    return templates

# function to detect and complete bonuses
def detect_and_complete(image, templates):
    # convert to grayscale
    gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
    # create binary image
    board_bin = get_binary_image(gray)
    
    # final matrix
    final_bonus_matrix = np.zeros((16, 16), dtype=int)
    # aux matrix for verifying scores
    aux_score_matrix = np.zeros((16, 16))

    # calculate padding
    # img > 1700px (1600 board + margins) we have padding of 100
    # or just the simple board
    off_x = 100 if image.shape[1] > 1700 else 0
    off_y = 100 if image.shape[0] > 1700 else 0

    # search for 2 on the board
    tmpl = templates.get(2)
    if tmpl is not None:
        # template dimensions after crop
        w, h = tmpl.shape[::-1]
        
        # rotate the template on all sides
        for angle in [0, 90, 180, 270]:
            if angle == 0: t = tmpl
            else: t = cv.rotate(tmpl, {90: cv.ROTATE_90_CLOCKWISE, 
                                       180: cv.ROTATE_180, 
                                       270: cv.ROTATE_90_COUNTERCLOCKWISE}[angle])
            
            # apply template matching
            res = cv.matchTemplate(board_bin, t, cv.TM_CCOEFF_NORMED)
            # get all locations where res > threshold 0.7
            locs = np.where(res >= MATCH_THRESHOLD)
            
            # iterate through found points
            for pt in zip(*locs[::-1]):
                score = res[pt[1], pt[0]]
                # calculate center to find coordinates in matrix
                center_x = pt[0] + w // 2
                center_y = pt[1] + h // 2
                
                # transform from pixels to row/column
                col = (center_x - off_x) // 100
                row = (center_y - off_y) // 100
                
                # check if we are on the board
                if 0 <= row < 16 and 0 <= col < 16:
                    # if we find a better score than the existing one we update
                    if score > aux_score_matrix[row, col]:
                        aux_score_matrix[row, col] = score
                        # mark that we found bonus '2' in this cell
                        final_bonus_matrix[row, col] = 2



    # complete the rest of the board according to the rules
    # the board is divided into 4 quadrants of 8x8
    # corners where we check for 2
    quadrants = [(0, 0), (0, 8), (8, 0), (8, 8)]
    
    # the 2 configurations for the bonuses of 1
    
    # descending (\)
    diag_descending = [
        (1,2), (2,3), (3,4), (4,5), (5,6),
        (2,1), (3,2), (4,3), (5,4), (6,5)
    ]
    
    # ascending (/)
    diag_ascending = [
        (1,5), (2,4), (3,3), (4,2), (5,1),
        (2,6), (3,5), (4,4), (5,3), (6,2)
    ]

    print("Checking quadrants for bonuses.")
    
    # analyze each quadrant
    for (r_off, c_off) in quadrants:
        # check position (1, 1) in the quadrant
        check_row = r_off + 1
        check_col = c_off + 1
        
        # there are only 2 possible configurations for a quadrant
        
        # if '2' is at 1,1
        if final_bonus_matrix[check_row, check_col] == 2:
            # add 2 at 6,6
            final_bonus_matrix[r_off + 6, c_off + 6] = 2
            
            # bonuses = diagonal / 
            coords_one = diag_ascending 
            
        else:
            # if '2' is NOT at 1,1
            # positions of "2" are at 1,6 and 6,1
            final_bonus_matrix[r_off + 1, c_off + 6] = 2
            final_bonus_matrix[r_off + 6, c_off + 1] = 2
            
            # bonuses = diagonal \
            coords_one = diag_descending

        # after deciding the type of quadrant we insert the 1 bonuses 
        for (dr, dc) in coords_one:
            final_bonus_matrix[r_off + dr, c_off + dc] = 1



    # visual debug
    # img_viz = image.copy()
    
    # for r in range(16):
    #     for c in range(16):
    #         val = final_bonus_matrix[r, c]
    #         if val > 0:
    #             x = off_x + c * 100
    #             y = off_y + r * 100
                
    #             color = (0, 0, 255) if val == 2 else (255, 0, 0)
    #             text = f"{val}"
                
    #             cv.rectangle(img_viz, (x, y), (x+100, y+100), color, 4)
    #             cv.putText(img_viz, text, (x+30, y+70), 
    #                        cv.FONT_HERSHEY_SIMPLEX, 2, color, 5)

    # plt.figure(figsize=(9, 9))
    # plt.imshow(cv.cvtColor(img_viz, cv.COLOR_BGR2RGB))
    # plt.title("Bonus")
    # plt.axis('off')
    # plt.show()

    # return the complete matrix (16x16) with the bonus map
    return final_bonus_matrix

# Calculating final score functions

In [ ]:
def get_full_line_score(matrix, r, c, dr, dc):
# function that calculates the length of a continuous line in a direction
# matrix - current board matrix
# r, c = start position (current piece)
# dr, dc = search direction 

    rows, cols = 16, 16
    
    # step 1: go backwards (inverse direction) until we find an empty space
    # we do this to find the start of the line
    # if the piece is placed in the middle of a line, we need to know where the line starts
    cr, cc = r, c
    while 0 <= cr - dr < rows and 0 <= cc - dc < cols:
        # .strip() removes spaces, if string is empty it means no piece
        if matrix[cr - dr][cc - dc].strip() == '': break
        cr -= dr
        cc -= dc
    
    # found the end of the line
    start_r, start_c = cr, cc
    
    # step 2: go forward (in the given direction) and count how many pieces there are
    length = 0
    curr_r, curr_c = start_r, start_c
    while 0 <= curr_r < rows and 0 <= curr_c < cols:
        if matrix[curr_r][curr_c].strip() == '': break
        length += 1
        curr_r += dr
        curr_c += dc
        
    # step 3: generate a unique id for this line
    # format: axis_rowstart_colstart (ex: h_2_5)
    # used id to avoid double scoring
    # if we place 3 pieces that complete the same line,
    # the function will be called 3 times, but we must score the line only 
    # once per turn, that s why we need an id
    axis = 'H' if dr == 0 else 'V'
    line_id = f"{axis}_{start_r}_{start_c}"
    
    # return line length and id
    return length, line_id


# main function that compares two board states and calculates the score
def calculate_score_and_bonus(old_matrix, new_matrix, bonus_map):
    new_pieces = []
    
    # identify new pieces by comparing with old matrix
    for r in range(16):
        for c in range(16):
            new_code = new_matrix[r][c].strip()
            old_code = old_matrix[r][c].strip()
            
            # if we have a piece now and it was empty before, it means its a new piece
            if new_code != '' and old_code == '':
                new_pieces.append((r, c, new_code))
    
    # no more than 6 new pieces per turn 
    if len(new_pieces) > 6:
        new_pieces = new_pieces[:6]

    # if no piece was placed, score is 0
    if not new_pieces:
        return [], 0

    total_score  = 0
    # set to keep ids of already scored lines
    lines_already_scored = set()
    
    # iterate through each newly placed piece
    for (r, c, cod) in new_pieces:
        
        # check horizontal
        # direction (0, 1)
        len_h, id_h = get_full_line_score(new_matrix, r, c, 0, 1)
        
        # score only if a line has at least 2 pieces
        if len_h > 1: 
            # check not to have already scored the line
            # (when checking another piece from the same line)
            if id_h not in lines_already_scored:
                pct = len_h
                # if we have 6 pieces in line, give bonus 6 pts
                if len_h == 6: pct += 6 
                
                total_score += pct
                # mark as scored
                lines_already_scored.add(id_h)
        
        # check vertical
        # direction (1, 0)
        len_v, id_v = get_full_line_score(new_matrix, r, c, 1, 0)
        if len_v > 1:
            if id_v not in lines_already_scored:
                pct = len_v
                if len_v == 6: pct += 6 # bonus qwirkle
                
                total_score += pct
                lines_already_scored.add(id_v)
                
        # score bonuses
        # if the piece was placed on a bonus, add to score
        if bonus_map is not None:
            bonus_val = bonus_map[r][c]
            if bonus_val > 0:
                total_score += bonus_val
    
    return new_pieces, total_score

# function to write the result to a text file
def write_output_file(full_path, pieces, score):
    # sort pieces by row and column
    pieces_sorted = sorted(pieces, key=lambda x: (x[0], x[1]))
    
    # open the file with write mode
    with open(full_path, "w") as f:
        for r, c, cod in pieces_sorted:
            # write in format position code piece
            # r+1 start from 1
            # chr(65+c) convert to letter A,B,C...
            f.write(f"{r+1}{chr(65+c)} {cod}\n")
        
        # score 
        f.write(f"{score}")


# Start:

# load templates only once
tpl_pieces = get_pieces_templates()
tpl_bonuses = load_bonus_2()


# main function for the whole code
def run_game(game_id, folder_in, folder_out):
    # game_id = game number = 1,2,3...
    game_id = str(game_id)
    print(f"Start checking game {game_id}:")

    # load the initial state x_00
    # use the initial image for bonuses
    img_name_00 = f"{game_id}_00.jpg"
    path_00 = os.path.join(folder_in, img_name_00)
    
    bonus_map = np.zeros((16, 16), dtype=int)
    # initialize the previous board as empty
    previous_matrix = np.full((16, 16), '  ', dtype=object)

    if os.path.exists(path_00):
        print(f"Checking the initial configuration: {img_name_00}")
        # extract the board from the image
        img_00 = extract_game_table(path_00)
        if img_00 is not None:
            # if it worked - extract bonuses
            bonus_map = detect_and_complete(img_00, tpl_bonuses)
            # we have the bonus matrix now
            
            # detect the starting pieces and save them in previous_matrix
            # do this so we DON'T score them when processing move 01
            previous_matrix = find_all_pieces_on_board_and_color(img_00, tpl_pieces)
    else:
        print(f"Did not find the initial image {path_00}")

    # iterate through moves
    idx = 1
    while True:
        # build the file name
        img_name = f"{game_id}_{idx:02d}.jpg"
        path_img = os.path.join(folder_in, img_name)
        
        # if we can't find the next image - exit the loop = game over
        if not os.path.exists(path_img): 
            break 
        
        print(f"Checking move: {img_name}")
        
        # extract the board from the image
        img = extract_game_table(path_img)
        if img is None: 
            idx += 1
            continue
            
        # detect the current pieces on the board
        current_matrix = find_all_pieces_on_board_and_color(img, tpl_pieces)
        
        # compare the previous state with the current one to calculate the score
        # the function returns the list of new pieces and the total score of the move
        pieces_new, score = calculate_score_and_bonus(previous_matrix, current_matrix, bonus_map)
        
        # write the result to a text file
        output_file_name = f"{game_id}_{idx:02d}.txt"
        output_file_path = os.path.join(folder_out, output_file_name)
        
        write_output_file(output_file_path, pieces_new, score)
        
        # print the saved pieces and the score
        str_list = ", ".join([f"{r+1}{chr(65+c)}:{k}" for r,c,k in pieces_new])
        print(f"    Saved: {str_list} | Score: {score}")
        
        # update
        # the current matrix becomes the previous matrix for the next turn
        previous_matrix = current_matrix.copy()
        idx += 1

    print(f"Finished game {game_id}.\n")

# 'Main' cell of the code

In [ ]:
# Folders:
INPUT_FOLDER = "antrenare"                      # folder with input jpg files
OUTPUT_FOLDER = "461_Lungu_Bogdan_Cosmin"       # folder with output text files 

# check output folder
if not os.path.exists(OUTPUT_FOLDER): 
    os.makedirs(OUTPUT_FOLDER)

# The path for the templates folder can be found in 2 locations:
# 1 The get_pieces_templates function in Piecees placeement/color detecting logic cell 3 line 12.
# 2 The load_bonus_2 function in Bonus detection and completion logic cell 4 line 28.

# Run modes:

# Single game ('1' represents the digit at the beginning of the series of images for a game, ex: 1_00.jpg / x_00.jpg)
# run_game(1, INPUT_FOLDER, OUTPUT_FOLDER)

# More games (1..5)
for i in range(1, 6): 
    run_game(i, INPUT_FOLDER, OUTPUT_FOLDER)
print("Done!")